# Anime Dubber v5 — TachiDUBB Edition (8-9/10) | Kaggle Free Optimized

Полный пайплайн с интеграцией **TachiDUBB** (MIT) — проверенные opensource компоненты:
1. **Extract Audio** — ffmpeg 48kHz stereo
2. **ASR** — faster-whisper `large-v3-turbo` (CUDA float16) → **TachiDUBB segment_post** (3 прохода: merge continuations → absorb micro → split long)
3. **Diarization** — pyannote 3.1 (опционально, HF_TOKEN)
4. **Translate** — Groq `qwen/qwen3.8-27b` → OpenRouter `llama-3.3-70b:free` → MyMemory
5. **Separation** — Demucs `htdemucs_ft` (CLI) → vocals + background
6. **TTS** — **Silero v4_ru** (оффлайн, 0 GPU-час в неделю) + Edge-TTS fallback | **TachiDUBB tts_qa** (Whisper-base roundtrip CER)
7. **Assemble** — **TachiDUBB assembler**: atempo-stretch (pitch сохранён), per-segment peak 0.7, 5ms cosine fades, loudnorm EBU R128, video extend
8. **Render** — ffmpeg merge (+ background track с ducking)

### GPU бюджет (T4 16GB, Kaggle Free 30ч/нед)
- Whisper large-v3-turbo: **~4GB** float16 → `del` + `empty_cache` после ASR
- Demucs htdemucs_ft: **~3GB** (только на этапе 5)
- Silero v4_ru: **~1GB** (CPU/GPU гибрид, не жрёт лимиты)
- Пик одновременно: **~5GB** — влазит с запасом, sequential loading
- TTS QA (whisper-base): **~1GB** — опционально, можно выключить

### Чекпоинты — resume при preemption (12ч лимит Kaggle)
Каждый этап пишет `JOB/checkpoints/*.json` — при перезапуске пропускается.

### API Keys (Kaggle Secrets → Add-ons)
- `GROQ_API_KEY` — console.groq.com/keys | `OPENROUTER_API_KEY` — openrouter.ai/keys | `HF_TOKEN` — huggingface.co/settings/tokens (опц.)

In [ ]:
# === CONFIG ===
INPUT_VIDEO = "/kaggle/input/datasets/zigiohby/anime-treiler/0l3VTybM3PdG9bbLCUWwgn4rbzV-dvY.mp4"
TARGET_LANG = "ru"
SOURCE_LANG = "ja"
ENABLE_DIARIZATION = True
ENABLE_TTS_QA = False  # True = Whisper-base проверка каждого сегмента (дольше, +1GB VRAM). Для 1:44 трейлера можно выключить.
QA_THRESHOLD = 0.4

# === SECRETS ===
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
OPENROUTER_API_KEY = secrets.get_secret("OPENROUTER_API_KEY")
try: HF_TOKEN = secrets.get_secret("HF_TOKEN")
except: HF_TOKEN = ""

# === INSTALL (кэшируется между рестартами, быстро) ===
!pip install -q faster-whisper demucs torch torchaudio soundfile numpy scipy pydub httpx edge-tts gTTS nest_asyncio 2>&1 | tail -1

import os, json, gc, re, subprocess, shutil, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

WORK = Path("/kaggle/working")
JOB = WORK / "dub_v5"
CKPT = JOB / "checkpoints"
CKPT.mkdir(parents=True, exist_ok=True)
(JOB / "tts").mkdir(exist_ok=True)

def ckpt_save(name, data):
    (CKPT / f"{name}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
def ckpt_load(name):
    p = CKPT / f"{name}.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None
def ckpt_exists(name):
    return (CKPT / f"{name}.json").exists()
def vram_cleanup():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except: pass

print(f"Input: {INPUT_VIDEO} exists={Path(INPUT_VIDEO).exists()}")
print(f"Job: {JOB}")
print(f"Diarization: {ENABLE_DIARIZATION and bool(HF_TOKEN)} | TTS QA: {ENABLE_TTS_QA}")
import torch; print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB" if torch.cuda.is_available() else "GPU: none")


In [ ]:
# === STAGE 1: Extract Audio ===
audio_path = JOB / "audio.wav"
if not audio_path.exists():
    subprocess.run(["ffmpeg","-y","-i",INPUT_VIDEO,"-vn","-acodec","pcm_s16le","-ar","48000","-ac","2",str(audio_path)], check=True, capture_output=True)
    print(f"Audio: {audio_path.stat().st_size/1024/1024:.2f} MB")
else:
    print(f"Audio cached: {audio_path.stat().st_size/1024/1024:.2f} MB")
# длительность видео — понадобится assembler'у для extend
def get_duration(p):
    r=subprocess.run(["ffprobe","-v","quiet","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",str(p)], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0
total_duration = get_duration(INPUT_VIDEO)
print(f"Video duration: {total_duration:.1f}s")

In [ ]:
# === STAGE 2: ASR + TachiDUBB segment_post (3 прохода) ===
import re
from faster_whisper import WhisperModel

# --- TachiDUBB segment_post.py (порт, без зависимостей) ---
_SENTENCE_END_RE = re.compile(r'[.!?"\']?\s*$')
_STRONG_BREAK_RE = re.compile(r'[.!?"\']?\s+')
def _merge_continuation_segments(segments, merge_gap=0.5, max_dur=12.0, max_chars=240):
    if len(segments)<2: return [dict(s) for s in segments]
    out=[dict(segments[0])]; joined=0
    for curr in segments[1:]:
        prev=out[-1]
        prev_spk, curr_spk = prev.get("speaker"), curr.get("speaker")
        same = (prev_spk==curr_spk) or not prev_spk or not curr_spk
        gap=curr["start"]-prev["end"]
        prev_text=(prev.get("text") or "").rstrip()
        unfinished=not _SENTENCE_END_RE.search(prev_text)
        combined_dur=curr["end"]-prev["start"]
        combined_chars=len(prev_text)+1+len((curr.get("text") or "").strip())
        if same and gap<=merge_gap and (unfinished or gap<0.2) and combined_dur<=max_dur and combined_chars<=max_chars:
            prev["end"]=curr["end"]; prev["text"]=(prev_text+" "+(curr.get("text") or "").lstrip()).strip()
            if "words" in prev and "words" in curr: prev["words"]=prev["words"]+curr["words"]
            joined+=1
        else: out.append(dict(curr))
    if joined: print(f"[post] Pass1 merged {joined} continuations")
    return out
def _absorb_micro_segments(segments, thr_sec=1.0, thr_chars=40):
    if len(segments)<2: return [dict(s) for s in segments]
    out=[]; absorbed=0; i=0
    segs=list(segments)
    while i<len(segs):
        seg=segs[i]; dur=seg["end"]-seg["start"]; txt=(seg.get("text") or "").strip()
        is_micro=dur<thr_sec and len(txt)<thr_chars
        if is_micro and out:
            prev=out[-1]; gap=seg["start"]-prev["end"]
            same=(prev.get("speaker")==seg.get("speaker") or not prev.get("speaker") or not seg.get("speaker"))
            if same and gap<1.5:
                prev["end"]=seg["end"]; prev["text"]=((prev.get("text") or "").rstrip()+" "+txt).strip()
                if "words" in prev and "words" in seg: prev["words"]+=seg["words"]
                absorbed+=1; i+=1; continue
        if is_micro and i+1<len(segs):
            nxt=segs[i+1]; gap=nxt["start"]-seg["end"]
            same=(nxt.get("speaker")==seg.get("speaker") or not nxt.get("speaker") or not seg.get("speaker"))
            if same and gap<1.5:
                merged=dict(nxt); merged["start"]=seg["start"]; merged["text"]=(txt+" "+(nxt.get("text") or "").lstrip()).strip()
                if "words" in seg and "words" in nxt: merged["words"]=seg["words"]+nxt["words"]
                segs[i+1]=merged; absorbed+=1; i+=1; continue
        out.append(dict(seg)); i+=1
    if absorbed: print(f"[post] Pass2 absorbed {absorbed} micro-segments")
    return out
def _split_very_long(segments, thr=15.0):
    out=[]; cnt=0
    for seg in segments:
        dur=seg["end"]-seg["start"]; txt=seg.get("text") or ""
        if dur<=thr or len(txt)<40: out.append(dict(seg)); continue
        bounds=[m.end() for m in _STRONG_BREAK_RE.finditer(txt) if 20<m.end()<len(txt)-20]
        if not bounds: out.append(dict(seg)); continue
        best=min(bounds, key=lambda b: abs(b-len(txt)/2)); frac=best/len(txt); t=seg["start"]+dur*frac
        a=dict(seg); a["end"]=t; a["text"]=txt[:best].strip()
        b=dict(seg); b["start"]=t; b["text"]=txt[best:].strip()
        if "words" in seg:
            n=len(seg["words"]); cut=int(n*frac); a["words"]=seg["words"][:cut]; b["words"]=seg["words"][cut:]
        out.extend([a,b]); cnt+=1
    if cnt: print(f"[post] Pass3 split {cnt} long segments")
    return out
def postprocess_segments(segs, merge_gap=0.5, max_merge_duration=12.0, max_merge_chars=240, split_threshold=15.0, micro_duration=1.0, micro_chars=40):
    if not segs: return segs
    n0=len(segs)
    segs=_merge_continuation_segments(segs, merge_gap, max_merge_duration, max_merge_chars)
    segs=_absorb_micro_segments(segs, micro_duration, micro_chars)
    segs=_split_very_long(segs, split_threshold)
    if len(segs)!=n0: print(f"[post] {n0} → {len(segs)} segments")
    return segs

# --- ASR ---
if ckpt_exists("asr"):
    seg_list = ckpt_load("asr")
    print(f"ASR cached: {len(seg_list)} segments")
else:
    model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
    segments, info = model.transcribe(str(audio_path), language=SOURCE_LANG, beam_size=5, word_timestamps=True, vad_filter=True, vad_parameters=dict(min_silence_duration_ms=500))
    raw = []
    for i,s in enumerate(segments):
        raw.append({"id": f"seg_{i:03d}", "start": s.start, "end": s.end, "text": s.text.strip(), "words": [{"word":w.word,"start":w.start,"end":w.end} for w in (s.words or [])]})
    print(f"ASR raw: {len(raw)} segments")
    # TachiDUBB cleanup — ДО перевода, чтобы LLM видел целые фразы
    seg_list = postprocess_segments(raw)
    # переназначаем id после пост-обработки
    for i,s in enumerate(seg_list): s["id"]=f"seg_{i:03d}"
    ckpt_save("asr", seg_list)
    del model; vram_cleanup()
    print(f"ASR done: {len(seg_list)} segments (после TachiDUBB postprocess)")

for s in seg_list[:3]: print(f"  {s['id']}: {s['start']:.2f}-{s['end']:.2f} {s['text'][:60]}")

In [ ]:
# === STAGE 3: Diarization (опционально, требует HF_TOKEN) ===
if ckpt_exists("diarized"):
    seg_list = ckpt_load("diarized")
    print(f"Diarization cached: {sorted(set(s.get('speaker','SPEAKER_00') for s in seg_list))}")
elif ENABLE_DIARIZATION and HF_TOKEN:
    try:
        from pyannote.audio import Pipeline
        pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=HF_TOKEN).to("cuda")
        diarization = pipeline(str(audio_path))
        for seg in seg_list:
            center=(seg["start"]+seg["end"])/2; seg["speaker"]="SPEAKER_00"
            for turn,_,spk in diarization.itertracks(yield_label=True):
                if turn.start <= center <= turn.end: seg["speaker"]=spk; break
        print(f"Diarization: {sorted(set(s['speaker'] for s in seg_list))}")
        ckpt_save("diarized", seg_list)
        del pipeline; vram_cleanup()
    except Exception as e:
        print(f"Diarization failed: {e} — fallback single speaker")
        for s in seg_list: s["speaker"]="SPEAKER_00"
        ckpt_save("diarized", seg_list)
else:
    for s in seg_list: s["speaker"]="SPEAKER_00"
    if not ckpt_exists("diarized"): ckpt_save("diarized", seg_list)
    print("Diarization: SKIPPED")

In [ ]:
# === STAGE 4: Translate (Groq qwen3.8 → OpenRouter → MyMemory) ===
import httpx
if ckpt_exists("translated"):
    seg_list = ckpt_load("translated")
    print(f"Translate cached: {len(seg_list)} lines")
    for s in seg_list[:3]: print(f"  {s['text'][:35]} -> {s.get('translation','')[:35]}")
else:
    def _parse_json_array(text, n):
        try:
            arr=json.loads(text)
            if len(arr)==n: return arr
        except: pass
        m=re.search(r'\[[\s\S]*?\]', text)
        if m:
            try:
                arr=json.loads(m.group())
                if len(arr)==n: return arr
            except: pass
        return []
    def translate_mymemory(texts, src, tgt):
        url="https://api.mymemory.translated.net/get"; res=[]
        for t in texts:
            try:
                r=httpx.get(url, params={"q":t,"langpair":f"{src}|{tgt}"}, timeout=10)
                r.raise_for_status(); tr=r.json()["responseData"]["translatedText"]
                res.append(tr if tr else t)
            except: res.append(t)
        return res
    def translate_texts(texts, src_lang, tgt_lang):
        if not texts: return texts
        names={"ja":"Japanese","en":"English","ko":"Korean","zh":"Chinese","ru":"Russian"}
        numbered="\n".join(f"{i+1}. {t}" for i,t in enumerate(texts))
        prompt=f"Translate the following {names.get(src_lang,src_lang)} anime dialogue lines to natural {names.get(tgt_lang,tgt_lang)}. Keep tone. Return ONLY JSON array, same order.\n\n{numbered}"
        try:
            with httpx.Client(timeout=60) as c:
                r=c.post("https://api.groq.com/openai/v1/chat/completions", headers={"Authorization": f"Bearer {GROQ_API_KEY}"}, json={"model":"qwen/qwen3.8-27b","messages":[{"role":"user","content":prompt}],"temperature":0.3,"max_tokens":4000})
                r.raise_for_status(); result=r.json()["choices"][0]["message"]["content"]
                arr=_parse_json_array(result,len(texts))
                if arr: print(f"✓ Groq qwen3.8: {len(arr)}"); return arr
        except Exception as e: print(f"Groq failed: {e}")
        try:
            with httpx.Client(timeout=60) as c:
                r=c.post("https://openrouter.ai/api/v1/chat/completions", headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"}, json={"model":"meta-llama/llama-3.3-70b-instruct:free","messages":[{"role":"user","content":prompt}]})
                r.raise_for_status(); result=r.json()["choices"][0]["message"]["content"]
                arr=_parse_json_array(result,len(texts))
                if arr: print(f"✓ OpenRouter: {len(arr)}"); return arr
        except Exception as e: print(f"OpenRouter failed: {e}")
        print("MyMemory fallback..."); return translate_mymemory(texts, src_lang, tgt_lang)
    texts=[s["text"] for s in seg_list]
    translations=translate_texts(texts, SOURCE_LANG, TARGET_LANG)
    # защита: если LLM вернул не тот размер — fallback на MyMemory поштучно
    if len(translations)!=len(texts):
        print(f"Bad translation count {len(translations)}!={len(texts)}, fallback..."); translations=translate_mymemory(texts, SOURCE_LANG, TARGET_LANG)
    for seg,tr in zip(seg_list, translations): seg["translation"]=tr; seg["translated_text"]=tr
    ckpt_save("translated", seg_list)
    print(f"Translated: {len(translations)}")
    for s in seg_list[:5]: print(f"  {s['text'][:35]} -> {s['translation'][:35]}")

In [ ]:
# === STAGE 5: Separation (Demucs HTDemucs) — с чекпоинтом ===
vocals_path = JOB / "vocals.wav"
background_path = JOB / "background.wav"
if vocals_path.exists() and background_path.exists():
    print(f"Separation cached: vocals {vocals_path.stat().st_size/1024/1024:.1f}MB, bg {background_path.stat().st_size/1024/1024:.1f}MB")
else:
    print("Demucs htdemucs (2 stems) — ~2-3 мин на T4...")
    r=subprocess.run(["python","-m","demucs","--two-stems","vocals","-n","htdemucs","-o",str(JOB),str(audio_path)], capture_output=True, text=True, timeout=600)
    if r.returncode!=0:
        print(r.stderr[-3000:]); raise RuntimeError("Demucs failed")
    out=JOB/"htdemucs"/audio_path.stem
    if out.exists():
        shutil.move(str(out/"vocals.wav"), str(vocals_path))
        shutil.move(str(out/"no_vocals.wav"), str(background_path))
        shutil.rmtree(str(out))
    vram_cleanup()
    print(f"Vocals {vocals_path.stat().st_size/1024/1024:.1f}MB | BG {background_path.stat().st_size/1024/1024:.1f}MB")

In [ ]:
# === STAGE 6: TTS — Silero v4_ru (оффлайн) + Edge-TTS fallback + TachiDUBB QA ===
import torch
tts_dir = JOB / "tts"
tts_dir.mkdir(exist_ok=True)

# Проверка кэша — валидируем что файлы реально существуют и >2KB, иначе регеним
cached = ckpt_load("tts_done")
_ok_cached = sum(1 for s in cached if Path(s.get("audio_path","")).exists() and Path(s["audio_path"]).stat().st_size>2000) if cached else 0
if cached and _ok_cached == len(cached) and _ok_cached>0 and all(s.get("translation") for s in cached):
    seg_list = cached
    print(f"TTS cached: {_ok_cached}/{len(cached)} files — пропускаем генерацию")
    print(f"Пример перевода: {seg_list[0].get('text','')[:30]} -> {seg_list[0].get('translation','')[:30]}")
else:
    if cached: print(f"TTS кэш битый ({_ok_cached}/{len(cached) if cached else 0}) — перегенерируем")
    print("Loading Silero v4_ru...")
    silero_model, _ = torch.hub.load(repo_or_dir='snakers4/silero-models', model='silero_tts', language='ru', speaker='v4_ru')
    silero_model.to("cuda" if torch.cuda.is_available() else "cpu")
    SILERO_SPEAKERS = ["kseniya","xenia","baya","aidar","eugene"]  # v4_ru valid, irina фикс
    EDGE_VOICES = ["ru-RU-DmitryNeural","ru-RU-SvetlanaNeural","ru-RU-YuriyNeural","ru-RU-DariyaNeural"]
    def synth_silero(text, speaker, out, sr=48000):
        clean=text.replace("…","...").replace("—","-").replace("“","\"").replace("”","\"").strip()
        if not clean: return False
        try:
            audio=silero_model.apply_tts(texts=[clean], speaker=speaker, sample_rate=sr, put_accent=True, put_yo=True)[0]
            import torchaudio; torchaudio.save(str(out), audio.unsqueeze(0), sr)
            return out.exists() and out.stat().st_size>1000
        except Exception as e: print(f"  silero err {speaker}: {e}"); return False
    qa_model=None
    def get_qa_model():
        global qa_model
        if qa_model is not None: return qa_model
        if not ENABLE_TTS_QA: return None
        try:
            from faster_whisper import WhisperModel
            qa_model=WhisperModel("base", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="float16" if torch.cuda.is_available() else "int8")
            return qa_model
        except Exception as e: print(f"QA load failed: {e}"); return None
    def cer(hyp, ref):
        if not ref: return 0.0 if not hyp else 1.0
        m,n=len(hyp),len(ref)
        if m==0 or n==0: return 1.0
        if m>n: hyp,ref=ref,hyp; m,n=n,m
        prev=list(range(m+1))
        for j in range(1,n+1):
            cur=[j]+[0]*m
            for i in range(1,m+1): cur[i]=min(prev[i]+1, cur[i-1]+1, prev[i-1]+(0 if hyp[i-1]==ref[j-1] else 1))
            prev=cur
        return prev[m]/max(n,1)
    def normalize(s): return re.sub(r"\s+"," ", re.sub(r"[^\w\s]"," ", s.lower(), flags=re.UNICODE)).strip()
    def check_qa(wav_path, target_text):
        m=get_qa_model()
        if m is None: return 0.0, "", {"cer":0}
        try:
            segs, info=m.transcribe(str(wav_path), language=TARGET_LANG, beam_size=1, vad_filter=False)
            hyp=" ".join(s.text.strip() for s in segs).strip()
            if not hyp: return 1.0, "", {"cer":1.0}
            c=cer(normalize(hyp), normalize(target_text))
            penalty=0.5 if (info.language!=TARGET_LANG and info.language_probability>0.8 and info.language in {"en","zh","ja","ko","ar"}) else 0
            return c+penalty, hyp, {"cer":round(c,3),"lang":info.language}
        except Exception as e: return 0.5, "", {"error":str(e)}
    print(f"Generating TTS for {len(seg_list)} segments...")
    for i, seg in enumerate(seg_list):
        out = tts_dir / f"{seg['id']}.wav"
        if out.exists() and out.stat().st_size>2000:
            seg["audio_path"]=str(out); seg["tts_path"]=str(out); continue
        text=seg.get("translation") or seg.get("translated_text") or seg["text"]
        if not text or (text.strip()==seg.get("text","").strip() and SOURCE_LANG!=TARGET_LANG):
            print(f"  WARN {seg['id']}: translation missing, using original: {text[:30]}")
        spk_idx = hash(seg.get("speaker","")) % len(SILERO_SPEAKERS) if seg.get("speaker") else i % len(SILERO_SPEAKERS)
        speaker=SILERO_SPEAKERS[spk_idx]
        ok=synth_silero(text, speaker, out)
        if not ok:
            voice=EDGE_VOICES[i % len(EDGE_VOICES)]
            mp3=tts_dir / f"{seg['id']}.mp3"
            try:
                import edge_tts, asyncio, nest_asyncio
                nest_asyncio.apply()
                async def _edge_save():
                    comm = edge_tts.Communicate(text.replace("…","...").replace("—","-"), voice)
                    await comm.save(str(mp3))
                # proven pattern: asyncio.run per segment (works on Kaggle, no nest_asyncio needed)
                asyncio.run(_edge_save())
                if mp3.exists() and mp3.stat().st_size>1000:
                    subprocess.run(["ffmpeg","-y","-i",str(mp3),"-ar","48000","-ac","1",str(out)], capture_output=True, timeout=30)
                    mp3.unlink(missing_ok=True)
                    ok=out.exists() and out.stat().st_size>1000
                    if ok: print(f"  edge fallback OK {seg['id']} {voice}")
                    else:
                        # retry with Dmitry (most stable) if Yuriy/Dariya failed
                        try:
                            alt_voice="ru-RU-DmitryNeural" if "Yuriy" in voice or "Dariya" in voice else "ru-RU-SvetlanaNeural"
                            async def _retry():
                                comm=edge_tts.Communicate(text.replace("…","...").replace("—","-"), alt_voice)
                                await comm.save(str(mp3))
                            import nest_asyncio; nest_asyncio.apply()
                            import asyncio as _aio; _aio.run(_retry())
                            if mp3.exists() and mp3.stat().st_size>1000:
                                subprocess.run(["ffmpeg","-y","-i",str(mp3),"-ar","48000","-ac","1",str(out)], capture_output=True, timeout=30)
                                mp3.unlink(missing_ok=True)
                                ok=out.exists() and out.stat().st_size>1000
                                if ok: print(f"  edge retry OK {seg['id']} {alt_voice}")
                        except Exception as e2: print(f"  edge retry failed {seg['id']}: {e2}")
            except Exception as e:
                print(f"  edge fallback failed {seg['id']}: {e}")
                # ultimate fallback: try gTTS if edge blocked
                try:
                    from gtts import gTTS
                    gTTS(text=text, lang='ru').save(str(mp3))
                    subprocess.run(["ffmpeg","-y","-i",str(mp3),"-ar","48000","-ac","1",str(out)], capture_output=True, timeout=30)
                    mp3.unlink(missing_ok=True)
                    ok=out.exists() and out.stat().st_size>1000
                    if ok: print(f"  gTTS fallback OK {seg['id']}")
                except Exception as e2: print(f"  gTTS also failed: {e2}")
        if ok and ENABLE_TTS_QA:
            score, hyp, diag = check_qa(out, text)
            seg["qa_score"]=score; seg["qa_hyp"]=hyp
            if score>QA_THRESHOLD:
                print(f"  {seg['id']} QA bad CER={diag.get('cer')} lang={diag.get('lang')} — retry")
                alt=SILERO_SPEAKERS[(spk_idx+1)%len(SILERO_SPEAKERS)]
                if synth_silero(text, alt, out):
                    score2, hyp2, _ = check_qa(out, text)
                    if score2<score: seg["qa_score"]=score2; print(f"    retry OK {score:.2f}->{score2:.2f}")
        seg["audio_path"]=str(out) if out.exists() and out.stat().st_size>1000 else None
        seg["tts_path"]=seg["audio_path"]
        status="OK" if seg["audio_path"] else "FAIL"
        qa_str=f" QA={seg.get('qa_score',0):.2f}" if ENABLE_TTS_QA and "qa_score" in seg else ""
        print(f"  {seg['id']}: {status}{qa_str} speaker={speaker} {text[:40]}")
    del silero_model
    if qa_model is not None: del qa_model
    vram_cleanup()
    ckpt_save("tts_done", seg_list)
    ok=sum(1 for s in seg_list if s.get("audio_path"))
    print(f"TTS done: {ok}/{len(seg_list)}")
    if ok==0: print("!!! TTS FAIL: все 0 — проверь переводы и Silero логи выше !!!")
    else:
        import soundfile as sf
        for s in seg_list[:3]:
            ap=s.get("audio_path")
            if ap and Path(ap).exists():
                info=sf.info(ap)
                print(f"  check {s['id']}: {info.frames/info.samplerate:.2f}s {Path(ap).stat().st_size/1024:.0f}KB")


In [ ]:
# === STAGE 7: TachiDUBB Assembler — профессиональная сборка ===
import numpy as np, soundfile as sf
from scipy.signal import resample as scipy_resample

PER_SEGMENT_PEAK=0.7
LN_I=-16; LN_TP=-1.5; LN_LRA=11

def ffmpeg_run(cmd, desc="", timeout=300):
    r=subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if r.returncode!=0: raise RuntimeError(f"{desc} failed: {r.stderr[:600]}")
    return r

def loudnorm_inplace(wav_path):
    try:
        tmp=str(wav_path)+".ln.wav"
        af=f"adeclick,highpass=f=20,alimiter=limit=0.95:level=disabled:attack=5:release=50,loudnorm=I={LN_I}:TP={LN_TP}:LRA={LN_LRA}"
        ffmpeg_run(["ffmpeg","-y","-i",str(wav_path),"-af",af,"-ar","48000",tmp], "loudnorm+declick")
        os.replace(tmp, str(wav_path)); return True
    except Exception as e:
        print(f"loudnorm warn: {e}")
        try:
            tmp=str(wav_path)+".ln.wav"
            ffmpeg_run(["ffmpeg","-y","-i",str(wav_path),"-af",f"loudnorm=I={LN_I}:TP={LN_TP}:LRA={LN_LRA}","-ar","48000",tmp], "loudnorm-fallback")
            os.replace(tmp, str(wav_path)); return True
        except Exception as e2: print(f"loudnorm fallback failed: {e2}"); return False

def atempo_stretch(wav_path, speed):
    if abs(speed-1.0)<0.02: return wav_path
    out=str(wav_path)+f".{speed:.2f}x.wav"
    try:
        subprocess.run(["ffmpeg","-y","-i",str(wav_path),"-filter:a",f"atempo={speed:.3f}",out], check=True, capture_output=True, timeout=60)
        return out
    except: return wav_path

def assemble_dubbed_audio(segments, total_duration, output_path, bg_path=None, sample_rate=48000, apply_loudnorm=True):
    # оценка нужной длительности (русский длиннее японского ~20-30%)
    est_extra=0
    for seg in segments:
        ap=seg.get("audio_path")
        if ap and Path(ap).exists():
            try:
                info=sf.info(ap); tts_dur=info.frames/info.samplerate; slot=seg["end"]-seg["start"]
                if tts_dur>slot: est_extra+=(tts_dur-slot)
            except: pass
    target_duration=total_duration+min(est_extra,15.0)+1.0
    n_samples=int(target_duration*sample_rate)
    mix=np.zeros(n_samples, dtype=np.float32)
    valid=0; stretched=0; current_end=0.0
    for seg in segments:
        ap=seg.get("audio_path")
        if not ap or not Path(ap).exists(): continue
        try:
            slot=seg["end"]-seg["start"]
            info=sf.info(ap); tts_dur=info.frames/info.samplerate
            text=(seg.get("translated_text") or seg.get("translation") or "").lstrip()
            has_emotion=text.startswith("(") and ")" in text[:30]
            max_stretch=1.22 if has_emotion else 1.15
            stretched_path=ap
            if slot>0.2 and tts_dur>slot*1.05:
                speed=min(tts_dur/slot, max_stretch)
                if speed>1.02:
                    stretched_path=atempo_stretch(ap, speed); stretched+=1
            data, sr=sf.read(stretched_path, dtype="float32")
            if data.ndim>1: data=data.mean(axis=1)
            if sr!=sample_rate:
                ratio=sample_rate/sr; new_len=int(len(data)*ratio)
                data=np.interp(np.linspace(0,len(data)-1,new_len), np.arange(len(data)), data).astype(np.float32)
            peak=float(np.abs(data).max())
            if peak>0.01: data=data*(PER_SEGMENT_PEAK/peak)
            fade=int(0.005*sample_rate)
            if len(data)>fade*2:
                fi=0.5*(1-np.cos(np.linspace(0,np.pi,fade))); fo=fi[::-1]
                data[:fade]*=fi; data[-fade:]*=fo
            start=max(seg["start"], current_end); off=int(start*sample_rate)
            end=min(off+len(data), n_samples); length=end-off
            if length>0:
                mix[off:off+length]+=data[:length]; valid+=1; current_end=start+length/sample_rate
                seg["placed_start"]=float(start); seg["placed_end"]=float(current_end)
        except Exception as e: print(f"skip {seg.get('id')}: {e}")
    if stretched: print(f"Time-stretched {stretched}/{valid} segments (atempo pitch-preserved)")
    if current_end>0 and current_end+0.5<target_duration: mix=mix[:int((current_end+0.5)*sample_rate)]
    m=np.abs(mix).max()
    if m>1.0: mix=mix/m*0.95
    sf.write(str(output_path), mix, sample_rate, subtype="PCM_16")
    print(f"Assembled {valid} segments -> {output_path} ({len(mix)/sample_rate:.1f}s)")
    if apply_loudnorm and valid>0: loudnorm_inplace(output_path)
    return output_path

# Запуск сборки
dubbed_wav = JOB / "dubbed.wav"
# check if TTS newer than dubbed -> force reassemble
_tts_mtime = max((Path(s["audio_path"]).stat().st_mtime for s in seg_list if s.get("audio_path") and Path(s["audio_path"]).exists()), default=0)
_dub_mtime = dubbed_wav.stat().st_mtime if dubbed_wav.exists() else 0
if dubbed_wav.exists() and ckpt_exists("assembled") and _dub_mtime > _tts_mtime:
    print(f"Assembled cached: {dubbed_wav.stat().st_size/1024/1024:.1f}MB")

else:
    assemble_dubbed_audio(seg_list, total_duration, dubbed_wav, bg_path=background_path, apply_loudnorm=True)
    ckpt_save("assembled", {"path": str(dubbed_wav), "segments": seg_list})
    print(f"Dubbed wav: {dubbed_wav.stat().st_size/1024/1024:.1f}MB")

In [ ]:
# === STAGE 8: Merge video + dubbed audio + background — динамический ducking ===
# Если Demucs вырезал вокал идеально (как у тебя), фон чистый — не нужно глушить его постоянно на 0.12
# Делаем динамический ducking: фон 0.9 тихо → 0.55 во время речи с плавными атакой/релисом
final_video = JOB / "output.mp4"
mixed_wav = JOB / "mixed.wav"

def dur(p):
    r=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",str(p)], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0
v_dur=dur(INPUT_VIDEO); a_dur=dur(dubbed_wav)
need_extend=a_dur > v_dur + 0.3
print(f"Video {v_dur:.1f}s vs Dubbed {a_dur:.1f}s -> {'extend (freeze last frame)' if need_extend else 'no extend'}")

# --- Динамический микс: background + dubbed с sidechain-подобным приглушением только во время речи ---
import numpy as np, soundfile as sf

print("Dynamic ducking mix...")
bg_data, bg_sr = sf.read(str(background_path), dtype='float32')
if bg_data.ndim==1: bg_data = np.stack([bg_data, bg_data], axis=1)
dub_data, dub_sr = sf.read(str(dubbed_wav), dtype='float32')
if dub_data.ndim>1: dub_data = dub_data.mean(axis=1)
if bg_sr != 48000:
    new_len = int(len(bg_data)*48000/bg_sr)
    new_bg = np.zeros((new_len, bg_data.shape[1]), dtype=np.float32)
    for ch in range(bg_data.shape[1]):
        new_bg[:,ch] = np.interp(np.linspace(0, len(bg_data)-1, new_len), np.arange(len(bg_data)), bg_data[:,ch])
    bg_data = new_bg
    bg_sr = 48000
if dub_sr != 48000:
    new_len = int(len(dub_data)*48000/dub_sr)
    dub_data = np.interp(np.linspace(0, len(dub_data)-1, new_len), np.arange(len(dub_data)), dub_data).astype(np.float32)
    dub_sr = 48000
max_len = max(len(bg_data), len(dub_data))
if len(bg_data) < max_len:
    bg_data = np.pad(bg_data, ((0, max_len-len(bg_data)), (0,0)))
if len(dub_data) < max_len:
    dub_data = np.pad(dub_data, (0, max_len-len(dub_data)))

# envelope: 1.0 = фон громко, 0.55 = приглушен во время речи (-5dB)
DUCK_TARGET = 0.55  # 0.55 мягко; 1.0 = вообще не приглушать (т.к. вокал вырезан идеально)
ATTACK_MS, RELEASE_MS = 50, 300
attack = int(ATTACK_MS/1000*48000); release = int(RELEASE_MS/1000*48000)
env = np.ones(max_len, dtype=np.float32)
for seg in seg_list:
    s = seg.get("placed_start", seg["start"]); e = seg.get("placed_end", seg["end"])
    start, end = int(s*48000), int(e*48000)
    start, end = max(0,start), min(max_len, end)
    if end<=start: continue
    seg_env = np.ones(end-start, dtype=np.float32) * DUCK_TARGET
    if len(seg_env) > attack: seg_env[:attack] = np.linspace(1.0, DUCK_TARGET, attack)
    if len(seg_env) > release: seg_env[-release:] = np.linspace(DUCK_TARGET, 1.0, release)
    if len(seg_env) > attack+release: seg_env[attack:-release] = DUCK_TARGET
    env[start:end] = np.minimum(env[start:end], seg_env)
for ch in range(bg_data.shape[1]):
    bg_data[:,ch] *= env
mixed = bg_data.copy()
mixed[:,0] += dub_data * 0.95
mixed[:,1] += dub_data * 0.95
peak = np.abs(mixed).max()
if peak>0: mixed = mixed / peak * 0.98
sf.write(str(mixed_wav), mixed, 48000)
print(f"Mixed: {mixed_wav.stat().st_size/1024/1024:.1f}MB peak {peak:.2f} duck {DUCK_TARGET} (1.0=без приглушения)")

if need_extend:
    ext=a_dur - v_dur + 0.2
    tpad=f"tpad=stop_mode=clone:stop_duration={ext:.2f}"
    subprocess.run(["ffmpeg","-y","-i",INPUT_VIDEO,"-i",str(mixed_wav),"-filter_complex",f"[0:v]{tpad}[v]","-map","[v]","-map","1:a","-c:v","libx264","-preset","veryfast","-c:a","aac","-b:a","192k",str(final_video)], check=True)
else:
    subprocess.run(["ffmpeg","-y","-i",INPUT_VIDEO,"-i",str(mixed_wav),"-map","0:v","-map","1:a","-c:v","copy","-c:a","aac","-b:a","192k",str(final_video)], check=True)

print(f"Final: {final_video.stat().st_size/1024/1024:.1f} MB")
r=subprocess.run(["ffprobe","-v","error","-show_entries","stream=codec_type,codec_name","-of","csv=p=0",str(final_video)], capture_output=True, text=True)
print(r.stdout.strip())
from IPython.display import FileLink, HTML
display(FileLink(str(final_video)))
display(HTML(f'<a download="output.mp4" href="/files/kaggle/working/dub_v5/output.mp4">⬇ Скачать output.mp4</a> <a download="mixed.wav" href="/files/kaggle/working/dub_v5/mixed.wav">⬇ Скачать mixed.wav</a>'))
print("DUCK_TARGET=0.55 сейчас. Если хочешь ВООБЩЕ не приглушать фон — поменяй на 1.0 и перезапусти только эту ячейку.")
